# Dashcam Crash — train → ensemble → submission

Run All. Needs: GPU on, internet on (one-time backbone download), two inputs attached:
1. competition data (has `train_labels.csv`)
2. this repo folder (has `src/train.py`) — upload as a dataset, or the notebook's working dir

Output: `submission.csv` = 50/50 average of ResNet50 + ResNet18 fold-ensembles, each with TTA.

In [ ]:
import os, glob, sys

def find_root(marker):
    for base in ['/kaggle/input', '.']:
        for p in glob.glob(f'{base}/**/{marker}', recursive=True):
            return os.path.dirname(os.path.abspath(p))
    raise FileNotFoundError(marker)

DATA = find_root('train_labels.csv')
CODE = find_root('train.py')
os.environ['PYTHONPATH'] = CODE + ':' + os.environ.get('PYTHONPATH', '')
print('DATA =', DATA)
print('CODE =', CODE)
assert os.path.isdir(os.path.join(DATA, 'train'))
!nvidia-smi --query-gpu=name --format=csv,noheader

In [ ]:
# Backbone A: ResNet50 (main model). ~2h on a P100/T4.
!python3 $CODE/train.py --data "$DATA" --out models_r50 --backbone resnet50 --epochs 25 --num-workers 2

In [ ]:
!python3 $CODE/infer.py --data "$DATA" --models models_r50 --out submission_r50.csv --num-workers 2

In [ ]:
# Backbone B: ResNet18 (ensemble diversity). ~1h. Skip if GPU quota is tight — R50 alone is ~0.87.
!python3 $CODE/train.py --data "$DATA" --out models_r18 --backbone resnet18 --epochs 25 --num-workers 2

In [ ]:
!python3 $CODE/infer.py --data "$DATA" --models models_r18 --out submission_r18.csv --num-workers 2

In [ ]:
import pandas as pd

a = pd.read_csv('submission_r50.csv')
b = pd.read_csv('submission_r18.csv')
assert (a.clip_id == b.clip_id).all()
sub = a.copy()
sub['label'] = ((a.label + b.label) / 2).clip(1e-6, 1 - 1e-6)
sub.to_csv('submission.csv', index=False)

print('mean pred:', round(float(sub.label.mean()), 3), '(expect ~0.3-0.4: test has ~1/3 crashes)')
print(sub.label.describe().round(3).to_string())

!python3 $CODE/../scripts/check_submission.py --submission submission.csv --sample "$DATA/sample_submission.csv" 2>/dev/null || python3 scripts/check_submission.py --submission submission.csv --sample "$DATA/sample_submission.csv"